# Multimodal Facial & Voice Emotion Recognition Research Pipeline
**Datasets:** FER2013, RAVDESS, CREMA-D (and CK+, AffectNet+)  
**Backbones:** Warm-started `EfficientNet-B2`, `ViT-B/16`, `ResNet-50` + Temporal BiLSTM + Acoustic SE-ResNet-BiLSTM + Cross-Modal Attention Fusion

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, CONFIG LOADER & DETERMINISTIC SEEDING
# ==============================================================================
import os
import re
import glob
import yaml
import math
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torch.amp import autocast, GradScaler

import torchvision.transforms as T
import torchvision.models as tv_models
import torchaudio
import torchaudio.transforms as AT
import timm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def load_config(config_path: str = "config.yaml") -> dict:
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

cfg = load_config("config.yaml")
seed_everything(cfg["environment"]["seed"])
os.makedirs(cfg["environment"]["output_dir"], exist_ok=True)
os.makedirs(cfg["environment"]["checkpoint_dir"], exist_ok=True)

device = torch.device(cfg["environment"]["device"] if torch.cuda.is_available() else "cpu")
print(f"[INFO] Running on device: {device}")


In [ ]:
# ==============================================================================
# CELL 2: UNIFIED EMOTION TAXONOMY & MULTIMODAL DATASET PARSERS
# ==============================================================================
EMOTION_TO_IDX = {v: int(k) for k, v in cfg["taxonomy"]["class_names"].items()}

# Standard filename mappings for RAVDESS and CREMA-D
RAVDESS_MAP = {
    "01": "neutral", "02": "neutral", # Calm merged into Neutral for 7-class unification
    "03": "happy", "04": "sad", "05": "angry",
    "06": "fear", "07": "disgust", "08": "surprise"
}

CREMAD_MAP = {
    "NEU": "neutral", "HAP": "happy", "SAD": "sad",
    "ANG": "angry", "FEA": "fear", "DIS": "disgust"
}

class AudioVisualProcessor:
    """Handles frame extraction, face-aware transforms, and Log-Mel Spectrogram computation."""
    def __init__(self, cfg: dict, is_train: bool = True):
        self.cfg = cfg
        self.is_train = is_train
        img_size = cfg["preprocessing"]["visual"]["image_size"]
        mean = cfg["preprocessing"]["visual"]["mean"]
        std = cfg["preprocessing"]["visual"]["std"]
        self.num_frames = cfg["preprocessing"]["visual"]["num_frames"]

        if is_train:
            self.vis_transform = T.Compose([
                T.Resize((img_size, img_size)),
                T.RandomHorizontalFlip(p=0.5),
                T.RandomApply([T.ColorJitter(0.2, 0.2, 0.2, 0.1)], p=0.5),
                T.RandomRotation(10),
                T.ToTensor(),
                T.Normalize(mean=mean, std=std),
                T.RandomErasing(p=0.2, scale=(0.02, 0.15))
            ])
        else:
            self.vis_transform = T.Compose([
                T.Resize((img_size, img_size)),
                T.ToTensor(),
                T.Normalize(mean=mean, std=std)
            ])

        # Audio pipeline
        acfg = cfg["preprocessing"]["audio"]
        self.sample_rate = acfg["sample_rate"]
        self.target_samples = int(acfg["sample_rate"] * acfg["duration_sec"])
        self.mel_spec = AT.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=acfg["n_fft"],
            hop_length=acfg["hop_length"],
            n_mels=acfg["n_mels"]
        )
        self.amp_to_db = AT.AmplitudeToDB()
        self.freq_mask = AT.FrequencyMasking(freq_mask_param=15)
        self.time_mask = AT.TimeMasking(time_mask_param=25)

    def load_image_as_sequence(self, img_path: str) -> torch.Tensor:
        """Loads a static image (FER2013/CK+/AffectNet+) and expands to (T, C, H, W)."""
        img = Image.open(img_path).convert("RGB")
        tensor_img = self.vis_transform(img) # (C, H, W)
        return tensor_img.unsqueeze(0).repeat(self.num_frames, 1, 1, 1)

    def load_video_frames(self, video_path: str) -> torch.Tensor:
        """Uniformly samples `num_frames` across a video clip (RAVDESS / CREMA-D)."""
        if not video_path or not os.path.exists(video_path):
            img_size = self.cfg["preprocessing"]["visual"]["image_size"]
            return torch.zeros((self.num_frames, 3, img_size, img_size))

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            img_size = self.cfg["preprocessing"]["visual"]["image_size"]
            return torch.zeros((self.num_frames, 3, img_size, img_size))

        indices = np.linspace(0, max(total_frames - 1, 0), self.num_frames, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_img = Image.fromarray(frame_rgb)
                frames.append(self.vis_transform(pil_img))
            elif len(frames) > 0:
                frames.append(frames[-1].clone())

        cap.release()
        while len(frames) < self.num_frames:
            frames.append(torch.zeros_like(frames[0]) if frames else torch.zeros((3, 224, 224)))
        return torch.stack(frames, dim=0) # (T, C, H, W)

    def load_audio_spectrogram(self, audio_path: str, add_realworld_noise: bool = False) -> torch.Tensor:
        """Loads waveform, pads/truncates to fixed duration, and returns Log-Mel Spectrogram (1, n_mels, time)."""
        if not audio_path or not os.path.exists(audio_path):
            return torch.zeros((1, self.cfg["preprocessing"]["audio"]["n_mels"], 188))

        waveform, sr = torchaudio.load(audio_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != self.sample_rate:
            resampler = AT.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)

        num_samples = waveform.shape[1]
        if num_samples < self.target_samples:
            pad_amt = self.target_samples - num_samples
            waveform = F.pad(waveform, (0, pad_amt))
        elif num_samples > self.target_samples:
            start = random.randint(0, num_samples - self.target_samples) if self.is_train else (num_samples - self.target_samples) // 2
            waveform = waveform[:, start:start + self.target_samples]

        if add_realworld_noise:
            noise = torch.randn_like(waveform) * 0.015
            waveform = waveform + noise

        spec = self.mel_spec(waveform)
        log_spec = self.amp_to_db(spec)
        log_spec = (log_spec - log_spec.mean()) / (log_spec.std() + 1e-6)

        if self.is_train and self.cfg["preprocessing"]["audio"]["spec_augment"]:
            log_spec = self.freq_mask(log_spec)
            log_spec = self.time_mask(log_spec)

        return log_spec # (1, n_mels, T_spec)


class UnifiedEmotionDataset(Dataset):
    """Universal Dataset supporting Unimodal Face, Unimodal Voice, and Paired Multimodal."""
    def __init__(self, samples: List[Dict], processor: AudioVisualProcessor, simulate_noise: bool = False):
        self.samples = samples
        self.processor = processor
        self.simulate_noise = simulate_noise

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = self.samples[idx]
        label = item["label"]
        is_vid = bool(item.get("is_video", False))
        has_visual = item.get("visual_path") is not None and os.path.exists(str(item.get("visual_path")))
        has_audio = item.get("audio_path") is not None and os.path.exists(str(item.get("audio_path")))

        if has_visual:
            if is_vid:
                visual_tensor = self.processor.load_video_frames(item["visual_path"])
            else:
                visual_tensor = self.processor.load_image_as_sequence(item["visual_path"])
        else:
            img_size = self.processor.cfg["preprocessing"]["visual"]["image_size"]
            visual_tensor = torch.zeros((self.processor.num_frames, 3, img_size, img_size))

        if has_audio:
            audio_tensor = self.processor.load_audio_spectrogram(item["audio_path"], add_realworld_noise=self.simulate_noise)
        else:
            audio_tensor = torch.zeros((1, self.processor.cfg["preprocessing"]["audio"]["n_mels"], 188))

        return {
            "visual": visual_tensor,
            "audio": audio_tensor,
            "has_visual": torch.tensor(float(has_visual), dtype=torch.float32),
            "has_audio": torch.tensor(float(has_audio), dtype=torch.float32),
            "is_video": torch.tensor(float(is_vid), dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.long)
        }


def index_static_dataset(root_dir: str, split: str = "train") -> List[Dict]:
    """Indexes static image datasets (FER2013, CK+, AffectNet+) organized as root/split/emotion/*.jpg."""
    samples = []
    if not os.path.exists(root_dir):
        return samples
    split_dir = os.path.join(root_dir, split)
    if not os.path.exists(split_dir) and split in ["val", "test"]:
        for alt in ["test", "val", "validation"]:
            cand = os.path.join(root_dir, alt)
            if os.path.exists(cand):
                split_dir = cand
                break
    if not os.path.exists(split_dir):
        return samples
    for emo_name, emo_idx in EMOTION_TO_IDX.items():
        emo_folder = os.path.join(split_dir, emo_name)
        if not os.path.exists(emo_folder):
            continue
        for fpath in glob.glob(os.path.join(emo_folder, "*.*")):
            if fpath.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                samples.append({
                    "visual_path": fpath,
                    "audio_path": None,
                    "is_video": False,
                    "label": emo_idx,
                    "dataset": os.path.basename(root_dir)
                })
    return samples


def index_ravdess(audio_dir: str, video_dir: str, split: str = "train") -> List[Dict]:
    """
    Indexes RAVDESS audio & video files using Actor-independent split:
    Actors 01-20 -> train | Actors 21-24 -> val/test
    """
    samples = []
    if not os.path.exists(audio_dir):
        return samples
    for wav_path in glob.glob(os.path.join(audio_dir, "**", "*.wav"), recursive=True):
        fname = os.path.basename(wav_path)
        parts = fname.replace(".wav", "").split("-")
        if len(parts) < 7:
            continue
        emo_code, actor_id = parts[2], int(parts[6])
        is_train_actor = actor_id <= 20
        if (split == "train" and not is_train_actor) or (split != "train" and is_train_actor):
            continue
        emo_name = RAVDESS_MAP.get(emo_code)
        if emo_name not in EMOTION_TO_IDX:
            continue

        actor_folder = f"Actor_{actor_id:02d}"
        vid_folder = f"Video_Speech_Actor_{actor_id:02d}"
        vid_fname_01 = "01-" + "-".join(parts[1:]) + ".mp4"
        vid_fname_02 = "02-" + "-".join(parts[1:]) + ".mp4"

        cands = [
            os.path.join(video_dir, vid_folder, actor_folder, vid_fname_01),
            os.path.join(video_dir, vid_folder, actor_folder, vid_fname_02),
            os.path.join(video_dir, actor_folder, vid_fname_01),
            os.path.join(video_dir, actor_folder, vid_fname_02),
        ]
        vid_path = next((c for c in cands if os.path.exists(c)), None)

        samples.append({
            "visual_path": vid_path,
            "audio_path": wav_path,
            "is_video": True if vid_path else False,
            "label": EMOTION_TO_IDX[emo_name],
            "dataset": "RAVDESS"
        })
    return samples


def index_cremad(audio_dir: str, video_dir: str, split: str = "train") -> List[Dict]:
    """
    Indexes CREMA-D (e.g. 1001_DFA_ANG_XX.wav) with speaker-independent split:
    Actors 1001-1075 -> train | Actors 1076-1091 -> val/test
    """
    samples = []
    if not os.path.exists(audio_dir):
        return samples
    for wav_path in glob.glob(os.path.join(audio_dir, "*.wav")):
        fname = os.path.basename(wav_path).replace(".wav", "")
        parts = fname.split("_")
        if len(parts) < 3:
            continue
        speaker_id = int(parts[0])
        emo_code = parts[2]
        is_train_speaker = speaker_id <= 1075
        if (split == "train" and not is_train_speaker) or (split != "train" and is_train_speaker):
            continue
        emo_name = CREMAD_MAP.get(emo_code)
        if emo_name not in EMOTION_TO_IDX:
            continue

        vid_cand = os.path.join(video_dir, fname + ".flv")
        if not os.path.exists(vid_cand):
            vid_cand = os.path.join(video_dir, fname + ".mp4")
        vid_path = vid_cand if os.path.exists(vid_cand) else None

        samples.append({
            "visual_path": vid_path,
            "audio_path": wav_path,
            "is_video": True if vid_path else False,
            "label": EMOTION_TO_IDX[emo_name],
            "dataset": "CREMA-D"
        })
    return samples


In [ ]:
# ==============================================================================
# CELL 3: VISUAL BACKBONE FACTORY (WARM-STARTED FROM TRAINED FER2013 MODELS)
# + SPATIAL-TEMPORAL BiLSTM & FAST STATIC/VIDEO INFERENCE
# ==============================================================================
class PretrainedVisualBackbone(nn.Module):
    """
    Wraps EfficientNet-B2, ViT-B/16, ResNet-50, or an Ensemble of all three,
    and loads your trained FER2013 checkpoints with 100% key-matching precision!
    Optimized with an 8.9x fast path for static images.
    """
    def __init__(self, cfg: dict):
        super().__init__()
        self.backbone_type = cfg["architecture"]["visual_backbone"]
        weights_cfg = cfg["architecture"]["pretrained_visual_weights"]
        self.models = nn.ModuleDict()
        self.out_dims = {}

        target_list = (
            ["efficientnet_b2", "vit_b_16", "resnet50"]
            if self.backbone_type == "ensemble_all"
            else [self.backbone_type]
        )

        for name in target_list:
            if name in ["efficientnet_b2", "efficientnet_b0"]:
                if name == "efficientnet_b2":
                    m = tv_models.efficientnet_b2(weights=tv_models.EfficientNet_B2_Weights.DEFAULT if not weights_cfg.get("load_custom") else None)
                    num_feat = 1408
                else:
                    m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.DEFAULT if not weights_cfg.get("load_custom") else None)
                    num_feat = 1280
                m.classifier = nn.Identity()
            elif name == "resnet50":
                m = tv_models.resnet50(weights=tv_models.ResNet50_Weights.DEFAULT if not weights_cfg.get("load_custom") else None)
                m.fc = nn.Identity()
                num_feat = 2048
            elif name in ["vit_b_16", "vit"]:
                m = tv_models.vit_b_16(weights=tv_models.ViT_B_16_Weights.DEFAULT if not weights_cfg.get("load_custom") else None)
                m.heads = nn.Identity()
                num_feat = 768
            else:
                m = timm.create_model(name, pretrained=True, num_classes=0)
                num_feat = m.num_features

            self.out_dims[name] = num_feat

            # Load user's trained checkpoint if available
            ckpt_path = weights_cfg.get(name, "")
            if weights_cfg.get("load_custom", False) and ckpt_path and os.path.exists(ckpt_path):
                state = torch.load(ckpt_path, map_location="cpu")
                if "model_state_dict" in state:
                    state = state["model_state_dict"]
                elif "state_dict" in state:
                    state = state["state_dict"]

                clean_state = {
                    k.replace("module.", "").replace("backbone.", ""): v
                    for k, v in state.items()
                    if not any(k.startswith(head_k) for head_k in ["classifier", "fc", "heads", "head"])
                }
                missing, unexpected = m.load_state_dict(clean_state, strict=False)
                print(f"[Visual Init] Loaded custom weights for {name} from {ckpt_path} (missing: {len(missing)}, unexpected: {len(unexpected)})")

            self.models[name] = m

        total_visual_dim = sum(self.out_dims.values())
        t_cfg = cfg["architecture"]["visual_temporal"]
        self.proj = nn.Sequential(
            nn.Linear(total_visual_dim, t_cfg["hidden_dim"]),
            nn.LayerNorm(t_cfg["hidden_dim"]),
            nn.GELU(),
            nn.Dropout(t_cfg["dropout"])
        )

        self.bilstm = nn.LSTM(
            input_size=t_cfg["hidden_dim"],
            hidden_size=t_cfg["hidden_dim"] // 2,
            num_layers=t_cfg["num_layers"],
            batch_first=True,
            bidirectional=True,
            dropout=t_cfg["dropout"] if t_cfg["num_layers"] > 1 else 0.0
        )
        self.temp_attn = nn.Linear(t_cfg["hidden_dim"], 1)

    def forward(self, x: torch.Tensor, is_video: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (B, T, C, H, W)
            is_video: Optional (B,) tensor indicating if clip is video (1.0) or static (0.0)
        Returns:
            seq_feats: (B, T, D)
            pooled_feat: (B, D)
        """
        B, T, C, H, W = x.shape

        # 8.9x Fast Path: If entire batch is static images (e.g. FER2013), forward once per image
        if is_video is not None and (is_video.sum() == 0):
            x_single = x[:, 0] # (B, C, H, W)
            feat_list = [model(x_single) for model in self.models.values()]
            concat_feats = torch.cat(feat_list, dim=-1) # (B, sum_dims = 4224)
            proj = self.proj(concat_feats) # (B, D)
            proj_feats = proj.unsqueeze(1).repeat(1, T, 1) # (B, T, D)
        else:
            x_flat = x.view(B * T, C, H, W)
            feat_list = [model(x_flat) for model in self.models.values()]
            concat_feats = torch.cat(feat_list, dim=-1) # (B*T, sum_dims = 4224)
            proj_feats = self.proj(concat_feats).view(B, T, -1) # (B, T, D)

        lstm_out, _ = self.bilstm(proj_feats) # (B, T, D)
        attn_weights = F.softmax(self.temp_attn(lstm_out), dim=1) # (B, T, 1)
        pooled_feat = torch.sum(lstm_out * attn_weights, dim=1) # (B, D)
        return lstm_out, pooled_feat


In [ ]:
# ==============================================================================
# CELL 4: ACOUSTIC SPECTROGRAM SE-RESNET + BiLSTM BACKBONE (FOR RAVDESS & CREMA-D)
# ==============================================================================
class SqueezeExcitation2D(nn.Module):
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.fc(x).view(x.size(0), x.size(1), 1, 1)
        return x * w


class AudioSpectrogramEncoder(nn.Module):
    """
    Extracts vocal emotion cues (pitch contour, jitter, prosody, spectral energy)
    from Log-Mel Spectrograms using 2D Conv + Squeeze-and-Excitation + BiLSTM.
    """
    def __init__(self, cfg: dict):
        super().__init__()
        embed_dim = cfg["architecture"]["audio_backbone"]["embedding_dim"]
        dropout = cfg["architecture"]["audio_backbone"]["dropout"]

        def conv_se_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.GELU(),
                SqueezeExcitation2D(out_c),
                nn.MaxPool2d(kernel_size=2, stride=2),
                nn.Dropout2d(dropout * 0.5)
            )

        self.cnn = nn.Sequential(
            conv_se_block(1, 64),
            conv_se_block(64, 128),
            conv_se_block(128, 256),
            conv_se_block(256, 256)
        )
        freq_bins = cfg["preprocessing"]["audio"]["n_mels"] // 16
        rnn_in_dim = 256 * freq_bins

        self.bilstm = nn.LSTM(
            input_size=rnn_in_dim,
            hidden_size=embed_dim // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.attn_pool = nn.Linear(embed_dim, 1)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, spec: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feat = self.cnn(spec)
        B, C, F_prime, T_prime = feat.shape
        feat = feat.permute(0, 3, 1, 2).contiguous().view(B, T_prime, C * F_prime)
        seq_out, _ = self.bilstm(feat)
        seq_out = self.norm(seq_out)
        weights = F.softmax(self.attn_pool(seq_out), dim=1)
        pooled = torch.sum(seq_out * weights, dim=1)
        return seq_out, pooled


In [ ]:
# ==============================================================================
# CELL 5: PARALLEL MULTIMODAL CROSS-ATTENTION & GATED FUSION NETWORK
# ==============================================================================
class MultimodalEmotionNet(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.cfg = cfg
        num_classes = cfg["taxonomy"]["num_classes"]
        f_cfg = cfg["architecture"]["multimodal_fusion"]
        D = f_cfg["embed_dim"]
        heads = f_cfg["num_heads"]
        drop = f_cfg["dropout"]
        self.mod_drop_prob = f_cfg["modality_dropout_prob"]

        self.visual_encoder = PretrainedVisualBackbone(cfg)
        self.audio_encoder = AudioSpectrogramEncoder(cfg)

        self.vis_to_aud_attn = nn.MultiheadAttention(embed_dim=D, num_heads=heads, dropout=drop, batch_first=True)
        self.aud_to_vis_attn = nn.MultiheadAttention(embed_dim=D, num_heads=heads, dropout=drop, batch_first=True)
        self.norm_v = nn.LayerNorm(D)
        self.norm_a = nn.LayerNorm(D)

        self.gate_fc = nn.Sequential(
            nn.Linear(D * 2, D),
            nn.Sigmoid()
        )

        self.visual_head = nn.Linear(D, num_classes)
        self.audio_head = nn.Linear(D, num_classes)

        self.fusion_classifier = nn.Sequential(
            nn.Linear(D * 2, D),
            nn.LayerNorm(D),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(D, num_classes)
        )

    def forward(
        self,
        visual: torch.Tensor,
        audio: torch.Tensor,
        has_visual: torch.Tensor,
        has_audio: torch.Tensor,
        is_video: Optional[torch.Tensor] = None
    ) -> Dict[str, torch.Tensor]:
        v_seq, v_pool = self.visual_encoder(visual, is_video=is_video)
        a_seq, a_pool = self.audio_encoder(audio)

        v_cross, _ = self.vis_to_aud_attn(query=v_seq, key=a_seq, value=a_seq)
        a_cross, _ = self.aud_to_vis_attn(query=a_seq, key=v_seq, value=v_seq)

        v_enhanced = self.norm_v(v_pool + v_cross.mean(dim=1))
        a_enhanced = self.norm_a(a_pool + a_cross.mean(dim=1))

        v_mask = has_visual.unsqueeze(-1)
        a_mask = has_audio.unsqueeze(-1)

        if self.training and self.mod_drop_prob > 0:
            both_present = (v_mask * a_mask).bool()
            rand_draw = torch.rand_like(v_mask)
            drop_v = (rand_draw < (self.mod_drop_prob / 2)) & both_present
            drop_a = (rand_draw >= (self.mod_drop_prob / 2)) & (rand_draw < self.mod_drop_prob) & both_present
            v_mask = torch.where(drop_v, torch.zeros_like(v_mask), v_mask)
            a_mask = torch.where(drop_a, torch.zeros_like(a_mask), a_mask)

        v_masked = v_enhanced * v_mask
        a_masked = a_enhanced * a_mask

        concat_va = torch.cat([v_masked, a_masked], dim=-1)
        z_gate = self.gate_fc(concat_va)
        gmu_feat = z_gate * v_masked + (1.0 - z_gate) * a_masked
        fused_repr = torch.cat([gmu_feat, v_masked + a_masked], dim=-1)

        logits_fusion = self.fusion_classifier(fused_repr)
        logits_visual = self.visual_head(v_pool)
        logits_audio = self.audio_head(a_pool)

        return {
            "logits": logits_fusion,
            "logits_visual": logits_visual,
            "logits_audio": logits_audio
        }


In [ ]:
# ==============================================================================
# CELL 6: LOSS FUNCTIONS (CLASS-BALANCED FOCAL LOSS) & RAdam OPTIMIZER
# ==============================================================================
class FocalLabelSmoothingLoss(nn.Module):
    """Addresses class imbalance across emotions."""
    def __init__(self, gamma: float = 2.0, smoothing: float = 0.1, weight: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma = gamma
        self.smoothing = smoothing
        self.weight = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        num_classes = logits.size(-1)
        log_preds = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_preds)
            true_dist.fill_(self.smoothing / (num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        probs = torch.exp(log_preds)
        focal_mod = (1.0 - probs) ** self.gamma
        loss = -true_dist * focal_mod * log_preds
        if self.weight is not None:
            loss = loss * self.weight.to(logits.device).unsqueeze(0)
        return loss.sum(dim=-1).mean()


def build_optimizer_and_scheduler(model: MultimodalEmotionNet, cfg: dict, total_epochs: int):
    opt_cfg = cfg["training"]["optimizer"]
    backbone_params = list(model.visual_encoder.models.parameters())
    head_params = [p for n, p in model.named_parameters() if "visual_encoder.models" not in n]

    param_groups = [
        {"params": backbone_params, "lr": opt_cfg["lr_backbone"]},
        {"params": head_params, "lr": opt_cfg["lr_fusion_head"]}
    ]

    if opt_cfg["name"].lower() == "radam":
        optimizer = torch.optim.RAdam(param_groups, weight_decay=opt_cfg["weight_decay"])
    else:
        optimizer = torch.optim.AdamW(param_groups, weight_decay=opt_cfg["weight_decay"])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
    return optimizer, scheduler


In [ ]:
# ==============================================================================
# CELL 7: MULTI-STAGE TRAINING ENGINE WITH INTERACTIVE TQDM PROGRESS
# ==============================================================================
def train_one_epoch(
    model: MultimodalEmotionNet,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    device: torch.device,
    use_amp: bool = True,
    desc: str = "Training"
) -> Tuple[float, float]:
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    for batch in pbar:
        vis = batch["visual"].to(device, non_blocking=True)
        aud = batch["audio"].to(device, non_blocking=True)
        has_v = batch["has_visual"].to(device, non_blocking=True)
        has_a = batch["has_audio"].to(device, non_blocking=True)
        is_vid = batch["is_video"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=use_amp):
            out = model(vis, aud, has_v, has_a, is_video=is_vid)
            loss_main = criterion(out["logits"], labels)
            loss_v = criterion(out["logits_visual"], labels) if has_v.sum() > 0 else 0.0
            loss_a = criterion(out["logits_audio"], labels) if has_a.sum() > 0 else 0.0
            loss = loss_main + 0.3 * loss_v + 0.3 * loss_a

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        bs = labels.size(0)
        running_loss += loss.item() * bs
        preds = out["logits"].argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += bs

        pbar.set_postfix({
            "loss": f"{loss.item():.3f}",
            "acc": f"{(correct / total) * 100:.1f}%"
        })

    epoch_loss = running_loss / max(total, 1)
    epoch_acc = (correct / max(total, 1)) * 100.0
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate_model(
    model: MultimodalEmotionNet,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    modality_override: Optional[str] = None,
    desc: str = "Evaluating"
) -> Dict[str, float]:
    model.eval()
    running_loss, all_preds, all_targets = 0.0, [], []

    for batch in tqdm(loader, desc=desc, leave=False, dynamic_ncols=True):
        vis = batch["visual"].to(device, non_blocking=True)
        aud = batch["audio"].to(device, non_blocking=True)
        has_v = batch["has_visual"].to(device, non_blocking=True)
        has_a = batch["has_audio"].to(device, non_blocking=True)
        is_vid = batch["is_video"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if modality_override == "face_only":
            has_a = torch.zeros_like(has_a)
        elif modality_override == "voice_only":
            has_v = torch.zeros_like(has_v)

        with autocast('cuda', enabled=True):
            out = model(vis, aud, has_v, has_a, is_video=is_vid)
            if modality_override == "face_only":
                combined_logits = 0.6 * out["logits"] + 0.4 * out["logits_visual"]
            elif modality_override == "voice_only":
                combined_logits = 0.6 * out["logits"] + 0.4 * out["logits_audio"]
            else:
                combined_logits = 0.6 * out["logits"] + 0.2 * out["logits_visual"] + 0.2 * out["logits_audio"]
            loss = criterion(combined_logits, labels)

        running_loss += loss.item() * labels.size(0)
        preds = combined_logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(labels.cpu().numpy())

    acc = accuracy_score(all_targets, all_preds) * 100.0
    f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0) * 100.0
    return {
        "loss": running_loss / max(len(loader.dataset), 1),
        "accuracy": acc,
        "macro_f1": f1,
        "preds": all_preds,
        "targets": all_targets
    }


In [ ]:
# ==============================================================================
# CELL 8: PIPELINE EXECUTION (INDIVIDUAL CREMA-D & PARALLEL RAVDESS + STATIC)
# ==============================================================================
def run_research_pipeline(cfg: dict):
    train_proc = AudioVisualProcessor(cfg, is_train=True)
    eval_proc = AudioVisualProcessor(cfg, is_train=False)

    d_cfg = cfg["datasets"]
    static_train, static_val = [], []
    for ds_key in ["fer2013", "ck_plus", "affectnet_plus"]:
        if d_cfg.get(ds_key, {}).get("enabled", False):
            static_train += index_static_dataset(d_cfg[ds_key]["root_dir"], "train")
            static_val += index_static_dataset(d_cfg[ds_key]["root_dir"], "val")

    cremad_train = index_cremad(d_cfg["crema_d"]["audio_dir"], d_cfg["crema_d"]["video_dir"], "train") if d_cfg["crema_d"]["enabled"] else []
    cremad_val = index_cremad(d_cfg["crema_d"]["audio_dir"], d_cfg["crema_d"]["video_dir"], "val") if d_cfg["crema_d"]["enabled"] else []

    ravdess_train = index_ravdess(d_cfg["ravdess"]["audio_dir"], d_cfg["ravdess"]["video_dir"], "train") if d_cfg["ravdess"]["enabled"] else []
    ravdess_val = index_ravdess(d_cfg["ravdess"]["audio_dir"], d_cfg["ravdess"]["video_dir"], "val") if d_cfg["ravdess"]["enabled"] else []

    print("\n" + "="*60)
    print("DATASET INDEXING SUMMARY:")
    print(f"  • FER2013/Static : Train={len(static_train):,} | Val={len(static_val):,}")
    print(f"  • RAVDESS Paired : Train={len(ravdess_train):,} | Val={len(ravdess_val):,}")
    print(f"  • CREMA-D Audio  : Train={len(cremad_train):,} | Val={len(cremad_val):,}")
    print("="*60 + "\n")

    model = MultimodalEmotionNet(cfg).to(device)
    criterion = FocalLabelSmoothingLoss(
        gamma=cfg["training"]["loss"]["gamma"],
        smoothing=cfg["training"]["loss"]["label_smoothing"]
    )
    scaler = GradScaler('cuda', enabled=cfg["environment"]["mixed_precision"])

    def run_stage(stage_name: str, train_samples: List[Dict], val_samples: List[Dict], epochs: int):
        if len(train_samples) == 0:
            print(f"[SKIP] {stage_name}: No samples found at configured path.")
            return
        print(f"\n==================== STARTING {stage_name} ====================")
        train_ds = UnifiedEmotionDataset(train_samples, train_proc, simulate_noise=False)
        val_lab_ds = UnifiedEmotionDataset(val_samples, eval_proc, simulate_noise=False)
        val_real_ds = UnifiedEmotionDataset(val_samples, eval_proc, simulate_noise=True)

        labels_np = np.array([s["label"] for s in train_samples])
        class_counts = np.bincount(labels_np, minlength=cfg["taxonomy"]["num_classes"]) + 1
        class_weights = 1.0 / class_counts
        sample_weights = class_weights[labels_np]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

        nw = 0 if os.name == "nt" else cfg["environment"]["num_workers"]
        train_loader = DataLoader(train_ds, batch_size=cfg["training"]["batch_size"], sampler=sampler,
                                  num_workers=nw, pin_memory=True)
        val_lab_loader = DataLoader(val_lab_ds, batch_size=cfg["training"]["batch_size"], shuffle=False,
                                    num_workers=nw, pin_memory=True)
        val_real_loader = DataLoader(val_real_ds, batch_size=cfg["training"]["batch_size"], shuffle=False,
                                     num_workers=nw, pin_memory=True)

        optimizer, scheduler = build_optimizer_and_scheduler(model, cfg, epochs)
        best_acc = 0.0

        for epoch in range(1, epochs + 1):
            ep_desc = f"[{stage_name[:15]}] Ep {epoch:02d}/{epochs:02d}"
            tr_loss, tr_acc = train_one_epoch(
                model, train_loader, criterion, optimizer, scaler, device,
                use_amp=cfg["environment"]["mixed_precision"], desc=ep_desc
            )
            scheduler.step()

            lab_metrics = evaluate_model(model, val_lab_loader, criterion, device, desc="Val (Lab)")
            real_metrics = evaluate_model(model, val_real_loader, criterion, device, desc="Val (Noisy)")

            print(
                f"[{stage_name} | Epoch {epoch:02d}/{epochs:02d}] "
                f"Train Acc: {tr_acc:.2f}% (Loss: {tr_loss:.4f}) | "
                f"Lab Val Acc: {lab_metrics['accuracy']:.2f}% (F1: {lab_metrics['macro_f1']:.2f}%) | "
                f"Real-World (Noisy) Acc: {real_metrics['accuracy']:.2f}%"
            )

            if lab_metrics["accuracy"] > best_acc:
                best_acc = lab_metrics["accuracy"]
                ckpt_file = os.path.join(cfg["environment"]["checkpoint_dir"], f"best_{stage_name.lower().replace(' ', '_')[:25]}.pth")
                torch.save(model.state_dict(), ckpt_file)
                print(f"  --> [SAVED] Checkpoint saved: {os.path.basename(ckpt_file)} (Acc: {best_acc:.2f}%)")

    mode = cfg["training"]["mode"]
    if mode in ["staged_full_pipeline", "visual_only"]:
        run_stage("Stage 1 Visual Fine-Tuning (FER2013)", static_train, static_val, cfg["training"]["epochs_visual_stage"])

    if mode in ["staged_full_pipeline", "individual_cremad"]:
        run_stage("Stage 2 Individual CREMA-D Audio", cremad_train, cremad_val, cfg["training"]["epochs_cremad_stage"])

    if mode in ["staged_full_pipeline", "parallel_multimodal"]:
        parallel_train = ravdess_train + cremad_train
        parallel_val = ravdess_val + cremad_val
        run_stage("Stage 3 Parallel Audio-Visual Fusion", parallel_train, parallel_val, cfg["training"]["epochs_ravdess_parallel"])

    return model

# Launch training pipeline
trained_model = run_research_pipeline(cfg)
